# Clarkson II full cleaning and modeling-ready pipeline

This notebook performs the remaining preprocessing steps after timestamp validation:

1. Split streams at timestamp backsteps and reset pairing queues.
2. Recompute press/release timing features.
3. Remove unmatched and impossible timing pairs.
4. Segment long idle gaps into typing bursts.
5. Remove poor-quality bursts.
6. Create fixed-size, non-overlapping windows.
7. Create chronological early/middle/late splits.
8. Fit feature normalization on training windows only.
9. Generate genuine/impostor verification trials.
10. Write final quality reports.

Raw files and earlier normalized outputs are never modified.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict, deque
import csv
import gzip
import json
import math
import random
import statistics

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents]
                     if (p / 'clarkson2_preprocessing').is_dir()), HERE)
WORK_DIR = PROJECT_ROOT / 'clarkson2_preprocessing'
INPUT_DIR = WORK_DIR / 'outputs' / 'normalized_events'
OUTPUT_DIR = WORK_DIR / 'outputs' / 'final_clean_dataset'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configurable quality and sampling policy.
IDLE_GAP_SECONDS = 30.0
MIN_PAIRS_PER_BURST = 20
WINDOW_PAIRS = 50
WINDOW_STRIDE = 50
MIN_WINDOWS_PER_USER = 3
TRAIN_FRACTION = 0.60
ADAPT_FRACTION = 0.20
MAX_HOLD_SECONDS = 5.0
MAX_FLIGHT_SECONDS = 60.0
IMPOSTOR_USERS_PER_TARGET = 5
MAX_TRIAL_WINDOWS_PER_USER = 500
RANDOM_SEED = 228

input_files = sorted(INPUT_DIR.glob('*.tsv.gz'))
if not input_files:
    raise FileNotFoundError(f'No normalized files found in {INPUT_DIR}')

print('Input users:', len(input_files))
print('Output directory:', OUTPUT_DIR)

Input users: 100
Output directory: C:\Users\keyar\Documents\GradSem3\Biometric\Term Project\clarkson2_preprocessing\outputs\final_clean_dataset


## Feature and window schema

The pipeline uses timing-only window features. Key identities are retained in the event-level output for auditability but are not used in the summary feature vector.

Timestamp backsteps define repair boundaries. The event order inside each repaired segment is preserved; the stream is never globally sorted.

In [2]:
EVENT_FIELDS = [
    'user_id', 'segment_id', 'burst_id', 'pair_index',
    'press_timestamp_100ns', 'release_timestamp_100ns',
    'press_elapsed_seconds', 'release_elapsed_seconds', 'key',
    'hold_seconds', 'press_to_press_seconds',
    'release_to_press_seconds', 'inter_event_seconds'
]

WINDOW_FIELDS = [
    'user_id', 'window_id', 'segment_id', 'burst_id', 'window_index',
    'start_elapsed_seconds', 'end_elapsed_seconds', 'pair_count',
    'hold_mean', 'hold_std', 'hold_median', 'hold_q25', 'hold_q75',
    'press_to_press_mean', 'press_to_press_std',
    'release_to_press_mean', 'release_to_press_std',
    'inter_event_mean', 'inter_event_std', 'typing_rate', 'split'
]

NUMERIC_FEATURES = [
    'hold_mean', 'hold_std', 'hold_median', 'hold_q25', 'hold_q75',
    'press_to_press_mean', 'press_to_press_std',
    'release_to_press_mean', 'release_to_press_std',
    'inter_event_mean', 'inter_event_std', 'typing_rate'
]

def percentile(values, q):
    if not values:
        return None
    ordered = sorted(values)
    position = (len(ordered) - 1) * q
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    return ordered[lower] + (ordered[upper] - ordered[lower]) * (position - lower)

def finite_stats(values):
    values = [float(value) for value in values if value is not None and math.isfinite(float(value))]
    if not values:
        return {'mean': None, 'std': None, 'median': None, 'q25': None, 'q75': None}
    return {
        'mean': statistics.fmean(values),
        'std': statistics.pstdev(values) if len(values) > 1 else 0.0,
        'median': statistics.median(values),
        'q25': percentile(values, 0.25),
        'q75': percentile(values, 0.75),
    }

def safe_mean(values):
    values = [float(value) for value in values if value is not None and math.isfinite(float(value))]
    return statistics.fmean(values) if values else None

def safe_std(values):
    values = [float(value) for value in values if value is not None and math.isfinite(float(value))]
    return statistics.pstdev(values) if len(values) > 1 else (0.0 if values else None)

## Rebuild clean event pairs and segment typing bursts

At a timestamp backstep, all active presses are discarded as incomplete and a new segment begins. This prevents a release in a later segment from being paired with a press from an earlier segment.

Pairs with negative, overly long, or otherwise invalid timing values are excluded. Long idle gaps start a new burst.

In [3]:
def process_user(input_path, event_output_path):
    user_id = input_path.name.removesuffix('.tsv.gz')
    stats = Counter()
    active = defaultdict(deque)
    previous_event = None
    previous_press = None
    previous_release = None
    first_timestamp = None
    segment_id = 0
    burst_id = 0
    pair_index = 0
    last_valid_pair_timestamp = None

    with gzip.open(input_path, 'rt', encoding='utf-8', newline='') as source, \
         gzip.open(event_output_path, 'wt', encoding='utf-8', newline='') as destination:
        reader = csv.DictReader(source, delimiter='\t')
        writer = csv.DictWriter(destination, fieldnames=EVENT_FIELDS,
                                delimiter='\t', lineterminator='\n')
        writer.writeheader()

        for event_number, row in enumerate(reader, start=1):
            stats['input_events'] += 1
            try:
                timestamp = int(row['timestamp_100ns'])
                flag = row['flag']
                key = row['key']
            except (KeyError, TypeError, ValueError):
                stats['invalid_input_events'] += 1
                continue

            if first_timestamp is None:
                first_timestamp = timestamp
            elapsed_seconds = (timestamp - first_timestamp) / 10_000_000
            if previous_event is not None and timestamp < previous_event:
                stats['timestamp_backsteps'] += 1
                stats['discarded_presses_at_backstep'] += sum(len(queue) for queue in active.values())
                active.clear()
                segment_id += 1
                burst_id += 1
                previous_press = None
                previous_release = None
                last_valid_pair_timestamp = None
            inter_event_seconds = ((timestamp - previous_event) / 10_000_000
                                  if previous_event is not None else None)

            if flag == '0':
                stats['press_events'] += 1
                press_to_press = ((timestamp - previous_press) / 10_000_000
                                  if previous_press is not None else None)
                release_to_press = ((timestamp - previous_release) / 10_000_000
                                    if previous_release is not None else None)
                active[key].append((timestamp, press_to_press, release_to_press, inter_event_seconds))
                previous_press = timestamp
            elif flag == '1':
                stats['release_events'] += 1
                if not active[key]:
                    stats['unmatched_releases'] += 1
                else:
                    press_timestamp, press_to_press, release_to_press, event_interval = active[key].popleft()
                    hold_seconds = (timestamp - press_timestamp) / 10_000_000
                    if hold_seconds < 0:
                        stats['negative_holds'] += 1
                    elif hold_seconds > MAX_HOLD_SECONDS:
                        stats['long_holds'] += 1
                    elif release_to_press is not None and release_to_press > MAX_FLIGHT_SECONDS:
                        stats['long_flights'] += 1
                    else:
                        if last_valid_pair_timestamp is not None and timestamp - last_valid_pair_timestamp > IDLE_GAP_SECONDS * 10_000_000:
                            burst_id += 1
                        pair_index += 1
                        writer.writerow({
                            'user_id': user_id, 'segment_id': segment_id, 'burst_id': burst_id,
                            'pair_index': pair_index,
                            'press_timestamp_100ns': press_timestamp,
                            'release_timestamp_100ns': timestamp,
                            'press_elapsed_seconds': f'{(press_timestamp - first_timestamp) / 10_000_000:.7f}',
                            'release_elapsed_seconds': f'{elapsed_seconds:.7f}',
                            'key': key,
                            'hold_seconds': f'{hold_seconds:.7f}',
                            'press_to_press_seconds': '' if press_to_press is None else f'{press_to_press:.7f}',
                            'release_to_press_seconds': '' if release_to_press is None else f'{release_to_press:.7f}',
                            'inter_event_seconds': '' if event_interval is None else f'{event_interval:.7f}',
                        })
                        stats['valid_pairs'] += 1
                        last_valid_pair_timestamp = timestamp
            else:
                stats['invalid_flags'] += 1
            previous_event = timestamp

    stats['unmatched_presses'] = sum(len(queue) for queue in active.values())
    stats['bursts_before_filter'] = burst_id + 1
    stats['segments'] = segment_id + 1
    stats['user_id'] = user_id
    return dict(stats)

EVENT_OUTPUT_DIR = OUTPUT_DIR / 'clean_events'
EVENT_OUTPUT_DIR.mkdir(exist_ok=True)
event_summaries = []
for index, input_path in enumerate(input_files, start=1):
    user_id = input_path.name.removesuffix('.tsv.gz')
    output_path = EVENT_OUTPUT_DIR / f'{user_id}.clean_events.tsv.gz'
    event_summaries.append(process_user(input_path, output_path))
    if index % 10 == 0 or index == len(input_files):
        print(f'Processed {index}/{len(input_files)} users')

with (OUTPUT_DIR / 'event_cleaning_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(event_summaries, handle, indent=2)

print('Clean event files:', len(event_summaries))

Processed 10/100 users


Processed 20/100 users


Processed 30/100 users


Processed 40/100 users


Processed 50/100 users


Processed 60/100 users


Processed 70/100 users


Processed 80/100 users


Processed 90/100 users


Processed 100/100 users
Clean event files: 100


## Create fixed-size windows and filter short bursts

Windows are non-overlapping by default to reduce train/test contamination. Windows are built within one burst and never cross a burst or repair-segment boundary.

In [4]:
def make_window(user_id, rows, window_index):
    holds = [float(row['hold_seconds']) for row in rows]
    press_to_press = [float(row['press_to_press_seconds']) for row in rows if row['press_to_press_seconds']]
    release_to_press = [float(row['release_to_press_seconds']) for row in rows if row['release_to_press_seconds']]
    intervals = [float(row['inter_event_seconds']) for row in rows if row['inter_event_seconds']]
    hold_stats = finite_stats(holds)
    start = float(rows[0]['press_elapsed_seconds'])
    end = float(rows[-1]['release_elapsed_seconds'])
    duration = max(end - start, 1e-9)
    result = {
        'user_id': user_id,
        'window_id': f'{user_id}_w{window_index:06d}',
        'segment_id': rows[0]['segment_id'],
        'burst_id': rows[0]['burst_id'],
        'window_index': window_index,
        'start_elapsed_seconds': start,
        'end_elapsed_seconds': end,
        'pair_count': len(rows),
        'hold_mean': hold_stats['mean'],
        'hold_std': hold_stats['std'],
        'hold_median': hold_stats['median'],
        'hold_q25': hold_stats['q25'],
        'hold_q75': hold_stats['q75'],
        'press_to_press_mean': safe_mean(press_to_press),
        'press_to_press_std': safe_std(press_to_press),
        'release_to_press_mean': safe_mean(release_to_press),
        'release_to_press_std': safe_std(release_to_press),
        'inter_event_mean': safe_mean(intervals),
        'inter_event_std': safe_std(intervals),
        'typing_rate': len(rows) / duration,
        'split': '',
    }
    return result

WINDOW_OUTPUT = OUTPUT_DIR / 'windows'
WINDOW_OUTPUT.mkdir(exist_ok=True)
window_rows = []
window_quality = []
for event_path in sorted(EVENT_OUTPUT_DIR.glob('*.clean_events.tsv.gz')):
    user_id = event_path.name.removesuffix('.clean_events.tsv.gz')
    bursts = defaultdict(list)
    with gzip.open(event_path, 'rt', encoding='utf-8', newline='') as source:
        reader = csv.DictReader(source, delimiter='\t')
        for row in reader:
            bursts[(row['segment_id'], row['burst_id'])].append(row)
    user_window_count = 0
    short_bursts = 0
    for burst_key, rows in bursts.items():
        if len(rows) < MIN_PAIRS_PER_BURST:
            short_bursts += 1
            continue
        for start in range(0, len(rows) - WINDOW_PAIRS + 1, WINDOW_STRIDE):
            window_rows.append(make_window(user_id, rows[start:start + WINDOW_PAIRS], user_window_count))
            user_window_count += 1
    window_quality.append({'user_id': user_id, 'windows': user_window_count, 'short_bursts': short_bursts})

with (WINDOW_OUTPUT / 'windows_raw.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=WINDOW_FIELDS)
    writer.writeheader()
    writer.writerows(window_rows)
with (WINDOW_OUTPUT / 'window_quality.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['user_id', 'windows', 'short_bursts'])
    writer.writeheader()
    writer.writerows(window_quality)

print('Raw windows:', len(window_rows))
print('Users with minimum windows:', sum(row['windows'] >= MIN_WINDOWS_PER_USER for row in window_quality))

Raw windows: 79286
Users with minimum windows: 94


## Chronological splits and training-only normalization

Each eligible user is split by window order, not by random sampling. The scaler is fitted only on early/training windows and then applied unchanged to adaptation and test windows. Users with fewer than the minimum number of windows are excluded from the modeling-ready file and retained in the quality report.

In [5]:
by_user = defaultdict(list)
for row in window_rows:
    by_user[row['user_id']].append(row)
eligible_users = {user_id for user_id, rows in by_user.items() if len(rows) >= MIN_WINDOWS_PER_USER}
excluded_users = sorted(set(by_user) - eligible_users)

for user_id in eligible_users:
    rows = sorted(by_user[user_id], key=lambda row: int(row['window_index']))
    n = len(rows)
    train_end = max(1, int(n * TRAIN_FRACTION))
    adapt_end = max(train_end + 1, int(n * (TRAIN_FRACTION + ADAPT_FRACTION)))
    adapt_end = min(adapt_end, n - 1) if n > 2 else n
    for index, row in enumerate(rows):
        row['split'] = 'train' if index < train_end else ('adapt' if index < adapt_end else 'test')

train_rows = [row for user_id in eligible_users for row in by_user[user_id] if row['split'] == 'train']
means = {}
stds = {}
for feature in NUMERIC_FEATURES:
    values = [float(row[feature]) for row in train_rows if row[feature] is not None]
    means[feature] = statistics.fmean(values) if values else 0.0
    stds[feature] = statistics.pstdev(values) if len(values) > 1 else 1.0
    if stds[feature] == 0:
        stds[feature] = 1.0

normalized_fields = WINDOW_FIELDS + [f'z_{feature}' for feature in NUMERIC_FEATURES]
normalized_rows = []
for user_id in sorted(eligible_users):
    for row in sorted(by_user[user_id], key=lambda item: int(item['window_index'])):
        output = dict(row)
        for feature in NUMERIC_FEATURES:
            value = float(row[feature]) if row[feature] is not None else means[feature]
            output[f'z_{feature}'] = (value - means[feature]) / stds[feature]
        normalized_rows.append(output)

with (OUTPUT_DIR / 'windows_normalized.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=normalized_fields)
    writer.writeheader()
    writer.writerows(normalized_rows)
with (OUTPUT_DIR / 'scaler_train_only.json').open('w', encoding='utf-8') as handle:
    json.dump({'means': means, 'stds': stds, 'features': NUMERIC_FEATURES}, handle, indent=2)

print('Eligible users:', len(eligible_users))
print('Excluded users:', len(excluded_users))
print('Training windows:', sum(row['split'] == 'train' for row in normalized_rows))
print('Adaptation windows:', sum(row['split'] == 'adapt' for row in normalized_rows))
print('Test windows:', sum(row['split'] == 'test' for row in normalized_rows))

Eligible users: 94
Excluded users: 1
Training windows: 47529
Adaptation windows: 15860
Test windows: 15896


## Generate verification trials

A genuine trial uses a window from the enrolled user. An impostor trial uses a window from another user in the same split. The random seed is fixed, and the number of impostor users per target is capped to keep the output manageable.

In [6]:
rng = random.Random(RANDOM_SEED)
rows_by_split_user = defaultdict(list)
for row in normalized_rows:
    rows_by_split_user[(row['split'], row['user_id'])].append(row)

trial_fields = ['enrolled_user', 'sample_user', 'window_id', 'split', 'label']
trials = []
for split in ['train', 'adapt', 'test']:
    users = sorted(user_id for current_split, user_id in rows_by_split_user if current_split == split)
    for enrolled_user in users:
        genuine_rows = rows_by_split_user[(split, enrolled_user)][:MAX_TRIAL_WINDOWS_PER_USER]
        for row in genuine_rows:
            trials.append({'enrolled_user': enrolled_user, 'sample_user': enrolled_user,
                           'window_id': row['window_id'], 'split': split, 'label': 1})
        impostor_users = [user for user in users if user != enrolled_user]
        rng.shuffle(impostor_users)
        for sample_user in impostor_users[:IMPOSTOR_USERS_PER_TARGET]:
            impostor_rows = rows_by_split_user[(split, sample_user)][:MAX_TRIAL_WINDOWS_PER_USER]
            for row in impostor_rows:
                trials.append({'enrolled_user': enrolled_user, 'sample_user': sample_user,
                               'window_id': row['window_id'], 'split': split, 'label': 0})

with (OUTPUT_DIR / 'verification_trials.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=trial_fields)
    writer.writeheader()
    writer.writerows(trials)

print('Verification trials:', len(trials))
print('Genuine trials:', sum(row['label'] == 1 for row in trials))
print('Impostor trials:', sum(row['label'] == 0 for row in trials))

Verification trials: 322969
Genuine trials: 51831
Impostor trials: 271138


## Final reports

The reports document excluded users, split sizes, cleaning counts, and all parameters used by the pipeline.

In [7]:
with (OUTPUT_DIR / 'eligible_users.txt').open('w', encoding='utf-8') as handle:
    handle.write('\n'.join(sorted(eligible_users)))
with (OUTPUT_DIR / 'excluded_users.txt').open('w', encoding='utf-8') as handle:
    handle.write('\n'.join(excluded_users))

with (OUTPUT_DIR / 'pipeline_config.json').open('w', encoding='utf-8') as handle:
    json.dump({
        'idle_gap_seconds': IDLE_GAP_SECONDS,
        'min_pairs_per_burst': MIN_PAIRS_PER_BURST,
        'window_pairs': WINDOW_PAIRS,
        'window_stride': WINDOW_STRIDE,
        'min_windows_per_user': MIN_WINDOWS_PER_USER,
        'train_fraction': TRAIN_FRACTION,
        'adapt_fraction': ADAPT_FRACTION,
        'max_hold_seconds': MAX_HOLD_SECONDS,
        'max_flight_seconds': MAX_FLIGHT_SECONDS,
        'impostor_users_per_target': IMPOSTOR_USERS_PER_TARGET,
        'random_seed': RANDOM_SEED,
    }, handle, indent=2)

with (OUTPUT_DIR / 'final_quality_report.json').open('w', encoding='utf-8') as handle:
    json.dump({
        'input_users': len(input_files),
        'eligible_users': len(eligible_users),
        'excluded_users': excluded_users,
        'input_events': sum(row.get('input_events', 0) for row in event_summaries),
        'valid_pairs': sum(row.get('valid_pairs', 0) for row in event_summaries),
        'negative_holds_removed': sum(row.get('negative_holds', 0) for row in event_summaries),
        'long_holds_removed': sum(row.get('long_holds', 0) for row in event_summaries),
        'long_flights_removed': sum(row.get('long_flights', 0) for row in event_summaries),
        'unmatched_presses': sum(row.get('unmatched_presses', 0) for row in event_summaries),
        'unmatched_releases': sum(row.get('unmatched_releases', 0) for row in event_summaries),
        'raw_windows': len(window_rows),
        'normalized_windows': len(normalized_rows),
        'verification_trials': len(trials),
    }, handle, indent=2)

print('Final pipeline artifacts written to:', OUTPUT_DIR)

Final pipeline artifacts written to: C:\Users\keyar\Documents\GradSem3\Biometric\Term Project\clarkson2_preprocessing\outputs\final_clean_dataset


## Important limitations

- The 30-second idle threshold, timing limits, and 50-pair window size are explicit starting assumptions and should be tested in sensitivity analyses.
- Verification labels are protocol-generated: same-user windows are genuine and different-user windows are impostors.
- The scaler is training-only, but user identities occur across splits because this is a longitudinal verification experiment.
- The pipeline does not reconstruct typed text and does not add the dataset to the repository.